# **Building a (basic) Boolean Search Engine on a Real Collection**

## Costruzione Inverted Index

Nel laboratiorio viene preso da un dataset di 20 Newsgroup un in sklearn una sola categoria (comp.graphics). Si osserva che di questa categoria si hanno 584 documenti totali e che il testo di ognuni di essi è **piuttosto sporco (header, indirizzi email, metadata, numeri, punteggiatura, etc..) -> ha senso fare preprocessing prima della costruzione dell'indice**.

Ricorda sempre che fare preprocessing significa fare compressione lossy -> perdita di informazione potenzialmente utile. Per questo nei sistemi reali la progettazione della pipeline di preprocessing è una decisione importante.

**Il preprocessing può rimuovere informazione utile?** Sì, in particolare eliminando stopwords i possono perdere differenze importanti, come frasi tipo "not working" (senza not tutto si riduce a working). Eliminando la punteggiatura si possono perdere differenze importanti, come "C++" vs "C". Lo stemming inoltre, nonostante sia molto più efficiente della lemmatizzazione, porta alle parole a radici piuttosto grezze -> parole con significati del tutto diversi possono finire nella stessa radice (es. "university" e "universe" -> "univers").

Costruzione dell'inverted index post preprocessing:

```python
postings = defaultdict(list)

for doc_id, tokens in tokenized_documents.items():
    unique_tokens = set(tokens)  # evitiamo duplicati dello stesso docID nella stessa posting list. Ma è ridondante?
    for token in unique_tokens:
        postings[token].append(doc_id)

# Ordiniamo le posting lists per docID. Ma non sono già ordinati "by construction"? Nel nostro caso sì, quindi sta parte sotto di base non serviva, come vedremo nel lab 1_3
for token in postings:
    postings[token] = sorted(postings[token])
```

set(tokens) al contrario è stato necessario in quanto in un inverted index ogni documento deve apparire al più una volta per posting list di un termine

Document frequency molto importante perché permette di identificare i termini più o meno selettivi ed è utilissima per ottimizzare le query

```python
document_frequency = {term: len(doc_ids) for term, doc_ids in postings.items()}
```

Una volta costruito l'inverted index, può essere utile salvarlo sul file di modo da evitare di doverlo ricostruire ogni volta. Nel colab si salvano i terms come termini testuali e le postings come liste python, ma si ricorda che in sistemi reali si sfruttano strutture più efficienti con compressione spinta di modo da risparmiare quanto più spazio possibile (es. gamma encoding, variable byte encoding, etc..)


## Querying SEMPLICE
Si vuole supportare query booleane del tipo "place, place AND authority, place OR authority, place AND NOT authority, place OR authority AND welcome"

Vediamo un'implementazione semplice di cui poi discuteremo il limiti.

Quando un utente scrive una query, è necessario anzitutto fare il preprocessing della query usando la stessa pipeline usata per i documenti, in modo da poterla confrontare con l'inverted index.

In questa prima implementazione si implementano AND, OR e NOT usando set, anche se non è la più efficiente dal punto di vista algoritmico.

**In particolare, nei sistemi reali convertire continuamente liste in set costa molta memoria e tempo --> gli engine reali usano posting lists ordinate, eseguendo AND/OR usando merge (lineare)**

```python
all_doc_ids = set(tokenized_documents.keys())

def intersection(a, b):
    return sorted(list(set(a) & set(b)))


def union(a, b):
    return sorted(list(set(a) | set(b)))


def get_not(word, postings_index, all_doc_ids):
    posting_list = get_posting(word, postings_index)
    return sorted(list(all_doc_ids.difference(set(posting_list))))
```

Chiaramente è poi necessario costruire una funzione che sappia riconoscere AND, OR e NOT, preprocessi i termini e quindi sappia gestire correttamente le query. Nella prima implementazione del colab non si gestiscono ancora le parentesi degli operatori e quindi possibili precedenze.

L'operazione di più difficile gestione è il NOT, nel codice si crea una funzione per la sua gestione che memorizza temporaneamente tutti i docID relativi ad ogni termine negato nella query tramite get_not, per poi eseguire le altre operazioni di AND/OR insieme a questo set di docID negati. 

I passaggi sono quindi:
- preprocessare i termini della query
- gestione del NOT: per ogni termine negato, memorizzare i docID negati in un set temporaneo
- gestione di AND/OR: eseguire le operazioni di AND/OR sui termini della query, tenendo conto dei docID negati memorizzati in precedenza

**Limiti gravi dell'implementazione:**
- non gestisce le parentesi e quindi la precedenza degli operatori
- uso di set per AND/OR che è inefficiente in termini di tempo e memoria, si dovrebbero sfruttare le posting lists ordinate con merge
- gestione del NOT in modo semplice, ma non molto elegante
- non verifica che la query sia scritta correttamente (spell correction)